# XGBoost Grid Search

This notebook performs a basic experiment for hyperparameter tuning for XGBoost models using grid search.

In [ ]:
# Import libraries
import pandas as pd
from model import train_best_model

In [2]:

files = {'Dataset_1/2visit_CN_MCI.csv','Dataset_1/2visit_MCI_AD.csv', 'Dataset_1/3visit_CN_MCI.csv','Dataset_1/3visit_MCI_AD.csv', 'Dataset_1/4visit_CN_MCI.csv','Dataset_1/4visit_MCI_AD.csv'
         'Dataset_1/5visit_CN_MCI.csv','Dataset_1/5visit_MCI_AD.csv'}

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

In [ ]:
# Batch grid search across datasets and progression types, saving artifacts and reports
import os
from datetime import datetime
from model import train_best_model

# Datasets and their progression type
files = [
    ("Dataset_1/2visit_CN_MCI.csv", "MCI"),
    ("Dataset_1/2visit_MCI_AD.csv", "AD"),
    ("Dataset_1/3visit_CN_MCI.csv", "MCI"),
    ("Dataset_1/3visit_MCI_AD.csv", "AD"),
    ("Dataset_1/4visit_CN_MCI.csv", "MCI"),
    ("Dataset_1/4visit_MCI_AD.csv", "AD"),
    ("Dataset_1/5visit_CN_MCI.csv", "MCI"),
    ("Dataset_1/5visit_MCI_AD.csv", "AD"),
]

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

results_dir = "grid_results"
os.makedirs(results_dir, exist_ok=True)

for path, prog in files:
    try:
        df = pd.read_csv(path)
        base = os.path.splitext(os.path.basename(path))[0]
        csv_out = os.path.join(results_dir, f"{base}_{prog}_cv_scores.csv")
        model_base = f"{base}"
        print(f"\n=== Running grid search for {base} ({prog}) ===")
        model, cols = train_best_model(
            df,
            progression_type=prog,
            param_grid=param_grid,
            csv_path=csv_out,
            save_dir="saved_models",
            model_base_name=model_base,
            save_artifacts=True,
        )
    except Exception as e:
        print(f"Error processing {path}: {e}")


=== Running grid search for 2visit_CN_MCI (MCI) ===
Using StratifiedKFold with n_splits=5
Using StratifiedKFold with n_splits=5
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       209
           1       0.31      0.25      0.28        16

    accuracy                           0.91       225
   macro avg       0.63      0.60      0.61       225
weighted avg       0.90      0.91      0.90       225


ROC AUC Score: 0.7593
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       209
           1       0.31      0.25      0.28        16

    accuracy                           0.91       225
   macro avg       0.63      0.60      0.61       225
weighted avg       0.90      0.91      0.90       225


ROC AUC Score: 0.7593
Bootstrap classification metrics: attempted=1000, auc_valid=1000, auc_skipped=0

Bootstrap 95% CI (n=1000):
- Accuracy: 0.907

In [ ]:
from model import preprocess_data, create_delta_features
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTENC
import numpy as np


path, prog = "Dataset_1/5visit_MCI_AD.csv", "AD"
df = pd.read_csv(path)
processed_df, _, _ = preprocess_data(create_delta_features(df), prog)

X = processed_df.drop('target', axis=1)
y = processed_df['target']

# Identify categorical column indices BEFORE transforms
categorical_cols = ['SEX', 'NACCFAM', 'CVHATT', 'CVAFIB', 'DIABETES', 
                    'HYPERCHO', 'HYPERTEN', 'B12DEF', 'DEPD', 'ANX', 'NACCTBI', 'RACE']
categorical_cols = [col for col in categorical_cols if col in X.columns.tolist()]
cat_indices = [X.columns.get_loc(col) for col in categorical_cols]

# 1. Impute (use most_frequent to keep categoricals as ints)
imputer = SimpleImputer(strategy='most_frequent')
X_imputed = imputer.fit_transform(X)

# 2. Scale BEFORE SMOTENC (SMOTENC handles categorical cols internally)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# 3. Apply SMOTENC on scaled data
sm = SMOTENC(categorical_features=cat_indices, k_neighbors=3, random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_scaled, y)

# Check results
print(f"Total samples: {len(y_resampled)}")
print(f"Class 0: {(y_resampled == 0).sum()}")
print(f"Class 1: {(y_resampled == 1).sum()}")
df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['target'] = y_resampled
df_resampled['is_synthetic'] = ['Original'] * len(y) + ['SMOTE_Generated'] * (len(y_resampled) - len(y))
df_resampled.to_csv('smote_resampled_data.csv', index=False)
print(f"\nResampled data saved to 'smote_resampled_data.csv'")
print(f"Original samples: {len(y)}")
print(f"SMOTE generated samples: {len(y_resampled) - len(y)}")
print("\nSample of SMOTE-generated instances:")
print(df_resampled[df_resampled['is_synthetic'] == 'SMOTE_Generated'].head(10))

Total samples: 254
Class 0: 127
Class 1: 127

Resampled data saved to 'smote_resampled_data.csv'
Original samples: 138
SMOTE generated samples: 116

Sample of SMOTE-generated instances:
          SEX      EDUC   ALCOHOL  NACCFAM  CVHATT    CVAFIB  DIABETES  \
138 -0.929981 -1.140183 -0.204808  0.83887     0.0 -0.279508 -0.470360   
139  1.075291 -0.677870 -0.204808  0.83887     0.0 -0.279508 -0.470360   
140 -0.929981  0.212983 -0.204808  0.83887     0.0 -0.279508 -0.470360   
141 -0.929981 -0.733847 -0.204808  0.83887     0.0 -0.279508 -0.470360   
142 -0.929981 -0.930840 -0.204808  0.83887     0.0 -0.279508 -0.470360   
143  1.075291 -0.281641  3.433847  0.83887     0.0 -0.279508  2.126029   
144 -0.929981 -0.210307 -0.204808  0.83887     0.0 -0.279508 -0.470360   
145 -0.929981  0.560667 -0.204808  0.83887     0.0 -0.279508 -0.470360   
146  1.075291 -0.537461  1.786694  0.83887     0.0 -0.279508  2.126029   
147  1.075291 -0.930524  5.444951  0.83887     0.0 -0.279508  2.126029   


## Experiment #2 

What's changed: 
1. SMOTE for synthetic oversampling within training folds. 
2. Additional patients added to 5 visit cohort
3. GDS and categorical longitudinal. 
3. Hearing and vision variables changed. 
4. MMSE imputation corrected. 

In [ ]:
# Experiment #2: Dataset_2 with post-split MMSE imputation + SMOTENC
import importlib
import model as model_module
importlib.reload(model_module)
from model import train_best_model

import os
import pandas as pd

files = [
    ("Dataset_2/2visit_CN_MCI.csv", "MCI"),
    ("Dataset_2/2visit_MCI_AD.csv", "AD"),
    ("Dataset_2/3visit_CN_MCI.csv", "MCI"),
    ("Dataset_2/3visit_MCI_AD.csv", "AD"),
    ("Dataset_2/4visit_CN_MCI.csv", "MCI"),
    ("Dataset_2/4visit_MCI_AD.csv", "AD"),
    ("Dataset_2/5visit_CN_MCI.csv", "MCI"),
    ("Dataset_2/5visit_MCI_AD.csv", "AD"),
]

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

results_dir = "grid_results"
os.makedirs(results_dir, exist_ok=True)

for path, prog in files:
    try:
        df = pd.read_csv(path)
        base = os.path.splitext(os.path.basename(path))[0]
        csv_out = os.path.join(results_dir, f"{base}_{prog}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Running grid search for {base} ({prog}) ===")
        print(f"{'='*60}")
        model, cols = train_best_model(
            df,
            progression_type=prog,
            param_grid=param_grid,
            csv_path=csv_out,
            save_dir="saved_models_2",
            model_base_name=base,
            save_artifacts=True,
        )
    except Exception as e:
        import traceback
        print(f"Error processing {path}: {e}")
        traceback.print_exc()


=== Running grid search for 2visit_CN_MCI (MCI) ===
No MMSE NaN values found; skipping MMSE imputation.
Using StratifiedKFold with n_splits=5
Using StratifiedKFold with n_splits=5
SMOTE resampling: 900 -> 1670 training samples
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       209
           1       0.35      0.44      0.39        16

    accuracy                           0.90       225
   macro avg       0.65      0.69      0.67       225
weighted avg       0.91      0.90      0.91       225


ROC AUC Score: 0.6953
SMOTE resampling: 900 -> 1670 training samples
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       209
           1       0.35      0.44      0.39        16

    accuracy                           0.90       225
   macro avg       0.65      0.69      0.67       225
weighted avg       0.91      0.90      0.91       225


R

/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


MMSE imputation complete (train fit, test transformed).
Using StratifiedKFold with n_splits=5
Using StratifiedKFold with n_splits=5
SMOTE resampling: 470 -> 880 training samples
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96       111
           1       0.33      0.14      0.20         7

    accuracy                           0.93       118
   macro avg       0.64      0.56      0.58       118
weighted avg       0.91      0.93      0.92       118


ROC AUC Score: 0.9524
SMOTE resampling: 470 -> 880 training samples
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96       111
           1       0.33      0.14      0.20         7

    accuracy                           0.93       118
   macro avg       0.64      0.56      0.58       118
weighted avg       0.91      0.93      0.92       118


ROC AUC Score: 0.9524
Bootstrap: valid=1000, total_a

# Experiment 3
What's changed:
1. Reverters dropped from dataset. 
All other parameters remain the same. 

In [ ]:
# Experiment #3: Dataset_2_no_reverters with post-split MMSE imputation + SMOTENC
import importlib
import model as model_module
importlib.reload(model_module)
from model import train_best_model

import os
import pandas as pd

files = [
    ("Dataset_2_no_reverter/2visit_CN_MCI.csv", "MCI"),
    ("Dataset_2_no_reverter/2visit_MCI_AD.csv", "AD"),
    ("Dataset_2_no_reverter/3visit_CN_MCI.csv", "MCI"),
    ("Dataset_2_no_reverter/3visit_MCI_AD.csv", "AD"),
    ("Dataset_2_no_reverter/4visit_CN_MCI.csv", "MCI"),
    ("Dataset_2_no_reverter/4visit_MCI_AD.csv", "AD"),
    ("Dataset_2_no_reverter/5visit_CN_MCI.csv", "MCI"),
    ("Dataset_2_no_reverter/5visit_MCI_AD.csv", "AD"),
]

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

results_dir = "grid_results"
os.makedirs(results_dir, exist_ok=True)

for path, prog in files:
    try:
        df = pd.read_csv(path)
        base = os.path.splitext(os.path.basename(path))[0]
        csv_out = os.path.join(results_dir, f"{base}_{prog}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Running grid search for {base} ({prog}) ===")
        print(f"{'='*60}")
        model, cols = train_best_model(
            df,
            progression_type=prog,
            param_grid=param_grid,
            csv_path=csv_out,
            save_dir="saved_models_3",
            model_base_name=base,
            save_artifacts=True,
        )
    except Exception as e:
        import traceback
        print(f"Error processing {path}: {e}")
        traceback.print_exc()


=== Running grid search for 2visit_CN_MCI (MCI) ===
No MMSE NaN values found; skipping MMSE imputation.
Using StratifiedKFold with n_splits=5
SMOTE resampling: 900 -> 1670 training samples
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       209
           1       0.35      0.44      0.39        16

    accuracy                           0.90       225
   macro avg       0.65      0.69      0.67       225
weighted avg       0.91      0.90      0.91       225


ROC AUC Score: 0.6953
Bootstrap: valid=1000, total_attempts=1000, skipped=0 (1000/1000)

Bootstrap 95% CI (n=1000, valid_samples=1000/1000):
- Accuracy: 0.902 (CI: 0.862, 0.942)
- Precision (macro): 0.653 (CI: 0.548, 0.764)
- Recall (macro): 0.688 (CI: 0.560, 0.822)
- F1 (macro): 0.668 (CI: 0.551, 0.772)
- ROC AUC: 0.695 (CI: 0.514, 0.852)

Saved artifacts:
- Model:   saved_models_3/2visit_CN_MCI_model_MCI.pkl
- Scaler:  saved_models_3/2visit_CN_MCI_scale

/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


MMSE imputation complete (train fit, test transformed).
Using StratifiedKFold with n_splits=5
SMOTE resampling: 457 -> 880 training samples
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       111
           1       0.80      1.00      0.89         4

    accuracy                           0.99       115
   macro avg       0.90      1.00      0.94       115
weighted avg       0.99      0.99      0.99       115


ROC AUC Score: 0.9910
Bootstrap: valid=1000, total_attempts=1016, skipped=16 (1000/1016)

Bootstrap 95% CI (n=1000, valid_samples=1000/1016):
- Accuracy: 0.991 (CI: 0.974, 1.000)
- Precision (macro): 0.900 (CI: 0.700, 1.000)
- Recall (macro): 0.995 (CI: 0.986, 1.000)
- F1 (macro): 0.942 (CI: 0.779, 1.000)
- ROC AUC: 0.991 (CI: 0.972, 1.000)

Saved artifacts:
- Model:   saved_models_3/4visit_CN_MCI_model_MCI.pkl
- Scaler:  saved_models_3/4visit_CN_MCI_scaler_MCI.pkl
- Imputer: saved_models_3/4visit_CN_MCI

# Experiment 4
1. New cohort from NACC update
2. reverters dropped
3. BMI imputation fixed
4. age below 50 dropped
5. LOOCV instead of stratifiedKfold 


In [2]:
import importlib
import model as model_module
importlib.reload(model_module)
from model import train_best_model

import os
import pandas as pd

files = [
    ("datasets/Dataset_updated/5visit_CN_MCI.csv", "MCI"),
    ("datasets/Dataset_updated/5visit_MCI_AD.csv", "AD"),
]

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

results_dir = "grid_results"
os.makedirs(results_dir, exist_ok=True)

for path, prog in files:
    try:
        df = pd.read_csv(path)
        base = os.path.splitext(os.path.basename(path))[0]
        csv_out = os.path.join(results_dir, f"{base}_{prog}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Running grid search for {base} ({prog}) ===")
        print(f"{'='*60}")
        model, cols = train_best_model(
            df,
            progression_type=prog,
            param_grid=param_grid,
            csv_path=csv_out,
            save_dir="experiments/experiment_4",
            use_smote=True,
            cv_method='loocv',
            model_base_name=base,
            save_artifacts=True,
        )
    except Exception as e:
        import traceback
        print(f"Error processing {path}: {e}")
        traceback.print_exc()


=== Running grid search for 5visit_CN_MCI (MCI) ===
Fitting MMSE imputer on training set (346 samples) with 16 covariates...


/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


MMSE imputation complete (train fit, test transformed).
Using Leave-One-Out CV (346 iterations, use_smote=True)
SMOTE resampling: 346 -> 592 training samples
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95        74
           1       0.75      0.69      0.72        13

    accuracy                           0.92        87
   macro avg       0.85      0.83      0.84        87
weighted avg       0.92      0.92      0.92        87


ROC AUC Score: 0.9428
Bootstrap: valid=1000, total_attempts=1000, skipped=0 (1000/1000)

Bootstrap 95% CI (n=1000, valid_samples=1000/1000):
- Accuracy: 0.920 (CI: 0.851, 0.966)
- Precision (macro): 0.848 (CI: 0.719, 0.974)
- Recall (macro): 0.826 (CI: 0.695, 0.945)
- F1 (macro): 0.837 (CI: 0.707, 0.938)
- ROC AUC: 0.943 (CI: 0.877, 0.991)

Saved artifacts:
- Model:   experiments/experiment_4/5visit_CN_MCI_model_MCI.pkl
- Scaler:  experiments/experiment_4/5visit_CN_MCI_scaler_MCI.pkl
- 

Above code showed LOOCV worked but was still evaluating on same size as StratifiedKfold. 

In [3]:
import importlib
import model as model_module
importlib.reload(model_module)
from model import train_best_model

import os
import pandas as pd

files = [
    ("datasets/Dataset_updated/4visit_CN_MCI.csv", "MCI"),
    ("datasets/Dataset_updated/4visit_MCI_AD.csv", "AD"),
]

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

results_dir = "grid_results"
os.makedirs(results_dir, exist_ok=True)

for path, prog in files:
    try:
        df = pd.read_csv(path)
        base = os.path.splitext(os.path.basename(path))[0]
        csv_out = os.path.join(results_dir, f"{base}_{prog}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Running grid search for {base} ({prog}) ===")
        print(f"{'='*60}")
        model, cols = train_best_model(
            df,
            progression_type=prog,
            param_grid=param_grid,
            csv_path=csv_out,
            save_dir="experiments/experiment_4",
            use_smote=True,
            cv_method='loocv',
            model_base_name=base,
            save_artifacts=True,
        )
    except Exception as e:
        import traceback
        print(f"Error processing {path}: {e}")
        traceback.print_exc()


=== Running grid search for 4visit_CN_MCI (MCI) ===
Fitting MMSE imputer on training set (452 samples) with 16 covariates...


/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


MMSE imputation complete (train fit, test transformed).
Using Leave-One-Out CV (452 iterations, use_smote=True)
SMOTE resampling: 452 -> 870 training samples
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       110
           1       0.50      0.50      0.50         4

    accuracy                           0.96       114
   macro avg       0.74      0.74      0.74       114
weighted avg       0.96      0.96      0.96       114


ROC AUC Score: 0.9773
Bootstrap: valid=1000, total_attempts=1014, skipped=14 (1000/1014)

Bootstrap 95% CI (n=1000, valid_samples=1000/1014):
- Accuracy: 0.965 (CI: 0.930, 0.991)
- Precision (macro): 0.741 (CI: 0.487, 0.996)
- Recall (macro): 0.741 (CI: 0.487, 0.996)
- F1 (macro): 0.741 (CI: 0.489, 0.942)
- ROC AUC: 0.977 (CI: 0.941, 1.000)

Saved artifacts:
- Model:   experiments/experiment_4/4visit_CN_MCI_model_MCI.pkl
- Scaler:  experiments/experiment_4/4visit_CN_MCI_scaler_MCI.pkl
-

/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


MMSE imputation complete (train fit, test transformed).
Using Leave-One-Out CV (180 iterations, use_smote=True)


KeyboardInterrupt: 

### Experiment 5

1. LOOCV, without SMOTE, MMSE imputer moved to inside of CV loop. 
2. Parrallelization added to use 10 cores to speed up.
3. Advanced logging. 

In [5]:
import importlib
import model as model_module
importlib.reload(model_module)
from model import train_best_model
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

files = [
    ("datasets/Dataset_updated/5visit_CN_MCI.csv", "MCI"),
    ("datasets/Dataset_updated/5visit_MCI_AD.csv", "AD"),
]

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

results_dir = "experiment_5/grid_results"
os.makedirs(results_dir, exist_ok=True)

exp5_results = {}  # Store results for visualization

for path, prog in files:
    try:
        df = pd.read_csv(path)
        base = os.path.splitext(os.path.basename(path))[0]
        csv_out = os.path.join(results_dir, f"{base}_{prog}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Running grid search for {base} ({prog}) ===")
        print(f"{'='*60}")
        model, cols = train_best_model(
            df,
            progression_type=prog,
            param_grid=param_grid,
            csv_path=csv_out,
            save_dir="experiments/experiment_5",
            use_smote=False,
            cv_method='loocv',
            n_jobs=10,
            model_base_name=base,
            save_artifacts=True,
        )
        # Store grid search scores for analysis
        exp5_results[f"{base}_{prog}"] = pd.read_csv(csv_out)
    except Exception as e:
        import traceback
        print(f"Error processing {path}: {e}")
        traceback.print_exc()

        fig, axes = plt.subplots(1, len(exp5_results), figsize=(7 * len(exp5_results), 5))
if len(exp5_results) == 1:
    axes = [axes]

### Visualizations ###

for ax, (label, scores_df) in zip(axes, exp5_results.items()):
    pivot = scores_df.groupby(['learning_rate', 'max_depth'])['avg AUC score'].mean().unstack()
    im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.2f}" for v in pivot.index])
    ax.set_xlabel('max_depth')
    ax.set_ylabel('learning_rate')
    ax.set_title(f'{label}\nMean AUC by LR × Depth')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Experiment 5 — LOOCV Mode A (No SMOTE): Grid Search AUC', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

for label, scores_df in exp5_results.items():
    top10 = scores_df.nlargest(10, 'avg AUC score')
    print(f"\n{'='*60}")
    print(f"Top 10 combos for {label}:")
    print(f"{'='*60}")
    print(top10.to_string(index=False))
    print(f"\nBest AUC: {top10['avg AUC score'].iloc[0]:.4f}")
    print(f"AUC range in top 10: {top10['avg AUC score'].iloc[0] - top10['avg AUC score'].iloc[-1]:.4f}")

hp_names = ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree']

for label, scores_df in exp5_results.items():
    fig, axes = plt.subplots(1, len(hp_names), figsize=(4 * len(hp_names), 4))
    fig.suptitle(f'{label} — AUC Distribution by Hyperparameter', fontsize=13)
    for ax, hp in zip(axes, hp_names):
        groups = [group['avg AUC score'].values for _, group in scores_df.groupby(hp)]
        labels = [str(v) for v in sorted(scores_df[hp].unique())]
        ax.boxplot(groups, tick_labels=labels)
        ax.set_xlabel(hp)
        ax.set_ylabel('AUC')
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()


=== Running grid search for 5visit_CN_MCI (MCI) ===

Mode A: LOOCV without SMOTE — full dataset (433 samples)
MMSE imputation: per-fold (fit on N-1)
Using Leave-One-Out CV (433 iterations x 243 combos, use_smote=False, n_jobs=10)


LOOCV grid search:   0%|          | 0/243 [00:00<?, ?combo/s]/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/.venv/l

KeyboardInterrupt: 